# Importing libraries

In [5]:
import json
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

# Input image path

In [6]:
input_path = r"C:\Users\Aayush Singh\OneDrive\Desktop\XN Project\IMG_1551.jpeg"

# Google cloud doc AI API credentials

In [ ]:
PROJECT_ID = "project-2802262d-c2f2-4c62-8a7"
LOCATION = "us"  # or "eu"
PROCESSOR_ID = "335e56c7120c9bf7"

PREPROCESSED_IMAGE = input_path
MIME_TYPE = "image/jpeg"

# Receipt processing function

In [8]:
def process_receipt():
    # Create client
    opts = ClientOptions(
        api_endpoint=f"{LOCATION}-documentai.googleapis.com"
    )

    client = documentai.DocumentProcessorServiceClient(
        client_options=opts
    )

    # Full processor path
    processor_name = client.processor_path(
        PROJECT_ID,
        LOCATION,
        PROCESSOR_ID
    )

    # Read image
    with open(PREPROCESSED_IMAGE, "rb") as f:
        image_content = f.read()

    # Create document object
    raw_document = documentai.RawDocument(
        content=image_content,
        mime_type=MIME_TYPE,
    )

    # Request
    request = documentai.ProcessRequest(
        name=processor_name,
        raw_document=raw_document,
    )

    # Process document
    result = client.process_document(request=request)

    document = result.document

    # Extract structured JSON
    receipt_json = extract_receipt_data(document)

    # ----------------------------------------
    # PRINT JSON OUTPUT HERE
    # ----------------------------------------
    print("\n========== RECEIPT JSON ==========\n")

    print(
        json.dumps(
            receipt_json,
            indent=2,
            ensure_ascii=False
        )
    )

    print("\n==================================\n")

    return receipt_json


# -----------------------------
# EXTRACT DATA
# -----------------------------
def extract_receipt_data(document):
    data = {
        "merchant_name": None,
        "transaction_date": None,
        "transaction_time": None,
        "currency": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "items": [],
        "raw_text": document.text,
    }

    print("\n========== RAW ENTITIES ==========\n")

    # Show all entities detected by Document AI
    for entity in document.entities:
        print(f"TYPE: {entity.type_}")
        print(f"VALUE: {entity.mention_text}")

        if entity.normalized_value:
            print(f"NORMALIZED: {entity.normalized_value.text}")

        print("--------------------------------")

        entity_type = entity.type_
        value = entity.mention_text

        normalized = None
        if entity.normalized_value:
            normalized = entity.normalized_value.text

        # -----------------------------
        # TOP LEVEL FIELDS
        # -----------------------------
        if entity_type == "supplier_name":
            data["merchant_name"] = normalized or value

        elif entity_type == "invoice_date":
            data["transaction_date"] = normalized or value

        elif entity_type == "total_amount":
            data["total"] = normalized or value

        elif entity_type == "subtotal_amount":
            data["subtotal"] = normalized or value

        elif entity_type == "total_tax_amount":
            data["tax"] = normalized or value

        elif entity_type == "currency":
            data["currency"] = value

        # -----------------------------
        # LINE ITEMS
        # -----------------------------
        elif entity_type == "line_item":
            item = {
                "description": None,
                "quantity": None,
                "unit_price": None,
                "total_price": None,
            }

            for prop in entity.properties:

                print(f"  PROPERTY TYPE: {prop.type_}")
                print(f"  PROPERTY VALUE: {prop.mention_text}")

                if prop.type_ == "line_item/description":
                    item["description"] = prop.mention_text

                elif prop.type_ == "line_item/quantity":
                    item["quantity"] = prop.mention_text

                elif prop.type_ == "line_item/unit_price":
                    item["unit_price"] = prop.mention_text

                elif prop.type_ == "line_item/amount":
                    item["total_price"] = prop.mention_text

            data["items"].append(item)

    print("\n==================================\n")

    return data


# -----------------------------
# RUN
# -----------------------------
if __name__ == "__main__":
    process_receipt()


========== RAW ENTITIES ==========

TYPE: total_tax_amount
VALUE: 0.52
NORMALIZED: 0.52
--------------------------------
TYPE: total_amount
VALUE: 8.01
NORMALIZED: 8.01
--------------------------------
TYPE: net_amount
VALUE: 7.49
NORMALIZED: 7.49
--------------------------------
TYPE: invoice_type
VALUE: 
NORMALIZED: restaurant_statement
--------------------------------
TYPE: currency
VALUE: $
NORMALIZED: USD
--------------------------------
TYPE: supplier_phone
VALUE: (617) 481-8635
NORMALIZED: +1 617 481 8635
--------------------------------
TYPE: supplier_website
VALUE: www.fiveguys.com/survey
--------------------------------
TYPE: supplier_address
VALUE: Store # MA-1816
1250 Hancock St
Quincy, MA 02169
NORMALIZED: 215 Thomas Burgin Pkwy Quincy, MA 02169 USA
--------------------------------
TYPE: invoice_date
VALUE: 4/23/2026
NORMALIZED: 2026-04-23
--------------------------------
TYPE: supplier_name
VALUE: 
NORMALIZED: Five Guys
--------------------------------
TYPE: line_item
VA